# Bronze Ingestion - Microsoft Planetary Computer (multi-collection)

**Workload:** Geohazard demo - Bronze layer  
**Source:** [Microsoft Planetary Computer STAC API](https://planetarycomputer.microsoft.com/api/stac/v1)  
**Area of Interest (AOI):** Pipeline-selected latitude, longitude, and catalog radius

## AOI behavior

Every pipeline notebook receives the same AOI so satellite, elevation, geology, soil, and overview outputs describe the same ground. The Microsoft Planetary Computer collections have broad coverage. The companion DataBC notebooks are British Columbia sources and can return empty tables outside their published coverage; an empty source is a data gap, not a measured zero.

## What this notebook does

1. Uses the selected AOI to calculate a WGS84 bounding box.
2. Provides one reusable `ingest_collection(...)` helper that queries the STAC API and writes item metadata, not rasters, into bronze Delta tables.
3. Ingests seven complementary collections, each into its own bronze table.

> This is a metadata-bronze pattern. It captures STAC item records, footprints, asset lists, and properties. Silver retrieves the underlying COG pixels only for the detailed analysis AOI.

## Collections ingested

| STAC collection | Bronze table | Theme | Time filter |
| --- | --- | --- | --- |
| `io-lulc-9-class` | `bronze_io_lulc_9_class` | Land use / land cover | none (annual) |
| `esa-worldcover` | `bronze_esa_worldcover` | Land cover | none (epochs) |
| `cop-dem-glo-30` | `bronze_cop_dem_glo_30` | 30 m elevation (DEM) | none (static) |
| `sentinel-2-l2a` | `bronze_sentinel_2_l2a` | Optical imagery | 2024 summer, cloud < 30% |
| `sentinel-1-rtc` | `bronze_sentinel_1_rtc` | Radar (SAR) | 2024 summer |
| `alos-palsar-mosaic` | `bronze_alos_palsar_mosaic` | L-band radar mosaic | none (annual) |
| `hgb` | `bronze_hgb` | Above/below-ground biomass | none (static) |

## 1. Shared configuration and helper
Run this cell once. It defines the AOI, the bounding box, a stable bronze
schema, and the `ingest_collection()` function used by every collection cell
below.

In [ ]:
# Pipeline parameters
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20

In [ ]:
import math
import json
import requests
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

if not -90.0 <= float(LATITUDE) <= 90.0:
    raise ValueError("LATITUDE must be between -90 and 90 degrees.")
if not -180.0 <= float(LONGITUDE) <= 180.0:
    raise ValueError("LONGITUDE must be between -180 and 180 degrees.")
if not 0.0 < float(RADIUS_KM) <= 100.0:
    raise ValueError("RADIUS_KM must be greater than 0 and no more than 100 km.")

AOI_NAME = f"AOI {float(LATITUDE):.4f}, {float(LONGITUDE):.4f}"
STAC_SEARCH_URL = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
SOURCE_API = "planetary_computer_stac"


def radius_to_bbox(lat, lon, radius_km):
    """Approximate a [min_lon, min_lat, max_lon, max_lat] box around a point."""
    lat_delta = radius_km / 111.32
    cos_lat = math.cos(math.radians(lat))
    lon_delta = 180.0 if abs(cos_lat) < 1e-8 else radius_km / (111.32 * cos_lat)
    return [lon - lon_delta, lat - lat_delta, lon + lon_delta, lat + lat_delta]


BBOX = radius_to_bbox(float(LATITUDE), float(LONGITUDE), float(RADIUS_KM))

# Explicit, stable schema so every collection lands in a consistently typed
# Delta table even when a particular property is missing for that collection.
BRONZE_SCHEMA = StructType([
    StructField("item_id", StringType(), True),
    StructField("collection", StringType(), True),
    StructField("datetime_utc", StringType(), True),
    StructField("bbox_minx", DoubleType(), True),
    StructField("bbox_miny", DoubleType(), True),
    StructField("bbox_maxx", DoubleType(), True),
    StructField("bbox_maxy", DoubleType(), True),
    StructField("asset_count", IntegerType(), True),
    StructField("asset_keys", StringType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("platform", StringType(), True),
    StructField("properties_json", StringType(), True),
    StructField("source_api", StringType(), True),
    StructField("query_lat", DoubleType(), True),
    StructField("query_lon", DoubleType(), True),
    StructField("query_radius_km", DoubleType(), True),
    StructField("query_datetime", StringType(), True),
    StructField("ingested_at_utc", TimestampType(), True),
])


def _num(value):
    """Best-effort float cast; returns None on failure."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def ingest_collection(collection, target_table, datetime_range=None,
                      max_items=12, query=None):
    """Query one Planetary Computer collection and write bronze item metadata."""
    payload = {"collections": [collection], "bbox": BBOX, "limit": max_items}
    if datetime_range:
        payload["datetime"] = datetime_range
    if query:
        payload["query"] = query

    response = requests.post(STAC_SEARCH_URL, json=payload, timeout=90)
    response.raise_for_status()
    features = response.json().get("features", [])

    # Static collections may ignore a datetime or property filter.
    if not features and (datetime_range or query):
        retry_payload = {"collections": [collection], "bbox": BBOX, "limit": max_items}
        response = requests.post(STAC_SEARCH_URL, json=retry_payload, timeout=90)
        response.raise_for_status()
        features = response.json().get("features", [])

    ingested_at = datetime.now(timezone.utc)
    rows = []
    for feature in features:
        properties = feature.get("properties", {}) or {}
        bounds = feature.get("bbox", [None, None, None, None]) or [None, None, None, None]
        assets = feature.get("assets", {}) or {}
        rows.append((
            str(feature.get("id")) if feature.get("id") is not None else None,
            str(feature.get("collection")) if feature.get("collection") is not None else collection,
            str(properties.get("datetime")) if properties.get("datetime") is not None else None,
            _num(bounds[0]) if len(bounds) > 0 else None,
            _num(bounds[1]) if len(bounds) > 1 else None,
            _num(bounds[2]) if len(bounds) > 2 else None,
            _num(bounds[3]) if len(bounds) > 3 else None,
            int(len(assets)),
            ",".join(sorted(assets.keys())) if assets else None,
            _num(properties.get("eo:cloud_cover")),
            str(properties.get("platform")) if properties.get("platform") is not None else None,
            json.dumps(properties, default=str),
            SOURCE_API,
            float(LATITUDE),
            float(LONGITUDE),
            float(RADIUS_KM),
            datetime_range,
            ingested_at,
        ))

    dataframe = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
    (dataframe.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    row_count = dataframe.count()
    print(f"[{collection}] -> {target_table}: wrote {row_count} rows")
    return row_count


print("AOI:", AOI_NAME)
print("BBOX (min_lon, min_lat, max_lon, max_lat):", [round(coordinate, 4) for coordinate in BBOX])
print("Helpers ready: radius_to_bbox(), ingest_collection(), BRONZE_SCHEMA")

## 2. `io-lulc-9-class` - Esri 10 m Land Use / Land Cover
Annual 9-class land-cover product. Useful for masking water, built-up areas,
and vegetation when interpreting geohazard signals.

In [ ]:
ingest_collection("io-lulc-9-class", "bronze_io_lulc_9_class", max_items=12)

## 3. `esa-worldcover` - ESA WorldCover 10 m
Global land-cover epochs (2020, 2021). A second, independent land-cover
reference to cross-check `io-lulc-9-class`.

In [ ]:
ingest_collection("esa-worldcover", "bronze_esa_worldcover", max_items=12)

## 4. `cop-dem-glo-30` - Copernicus DEM 30 m
Global digital elevation model. Slope and elevation derived from this DEM are
core inputs for landslide / geohazard susceptibility.

In [ ]:
ingest_collection("cop-dem-glo-30", "bronze_cop_dem_glo_30", max_items=12)

## 5. `sentinel-2-l2a` - Sentinel-2 surface reflectance
10 m optical imagery. Filtered to the 2024 snow-free season and to scenes with
**less than 30% cloud cover** using the STAC `query` extension.

In [ ]:
ingest_collection(
    "sentinel-2-l2a",
    "bronze_sentinel_2_l2a",
    datetime_range="2024-06-01/2024-09-30",
    max_items=20,
    query={"eo:cloud_cover": {"lt": 30}},
)

## 6. `sentinel-1-rtc` - Sentinel-1 Radiometrically Terrain Corrected
C-band radar that sees through cloud and night. Filtered to the same 2024
summer window for temporal alignment with Sentinel-2.

In [ ]:
ingest_collection(
    "sentinel-1-rtc",
    "bronze_sentinel_1_rtc",
    datetime_range="2024-06-01/2024-09-30",
    max_items=20,
)

## 7. `alos-palsar-mosaic` - ALOS PALSAR annual mosaic
L-band radar penetrates vegetation better than C-band and is sensitive to
surface roughness and moisture - complementary to Sentinel-1.

In [ ]:
ingest_collection("alos-palsar-mosaic", "bronze_alos_palsar_mosaic", max_items=12)

## 8. `hgb` - Harmonized Global Biomass
Above- and below-ground biomass. A vegetation-load proxy that helps separate
vegetated slopes from bare/disturbed ground.

In [ ]:
ingest_collection("hgb", "bronze_hgb", max_items=12)

## 9. Verification
Confirm every bronze table exists and report its row count. A `MISSING` line
means that collection cell has not been run yet (or returned no items).

In [ ]:
tables = [
    "bronze_io_lulc_9_class",
    "bronze_esa_worldcover",
    "bronze_cop_dem_glo_30",
    "bronze_sentinel_2_l2a",
    "bronze_sentinel_1_rtc",
    "bronze_alos_palsar_mosaic",
    "bronze_hgb",
]
for t in tables:
    try:
        c = spark.table(t).count()
        print(f"{t:30s} {c:6d} rows")
    except Exception as e:
        print(f"{t:30s} MISSING ({e.__class__.__name__})")